In [ ]:
# -*- coding: utf-8 -*-
import os
import sys
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
from obspy import read
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed # 🔥 TAMBAHAN UNTUK PARALELISASI

# ==============================================================================
# 🎛️ PARAMETER DIREKTORI UTAMA
# ==============================================================================
BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/rep_code"
MODEL_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20'
EMB_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909'

OUTPUT_WAVEFORM_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/output_waveform_indonesia_0109'
PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv'
PATH_DEMO_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/Code & Figure demo'

PATH_FINAL_REPORT = os.path.join(PATH_DEMO_DIR, 'mcu_quake_indonesia_true_3C_SCROLLING_MULTI_report.csv')

INPUT_SIZE = 700

if PATH_DEMO_DIR not in sys.path: sys.path.append(PATH_DEMO_DIR)
if BASE_REP not in sys.path: sys.path.append(BASE_REP)

try:
    from Library import utils, dataset
except ImportError:
    print("❌ Gagal memuat Library.")
    sys.exit()

# ==============================================================================
# 🔥 LOGIKA SCROLLING WINDOW 3C
# ==============================================================================
def sliding_window_inference_3c(data_z, data_n, data_e, model, pdf_3C):
    step_size = 100
    window_size = INPUT_SIZE
    
    if len(data_z) < window_size:
        data_z = np.pad(data_z, (0, window_size - len(data_z)), 'constant')
        data_n = np.pad(data_n, (0, window_size - len(data_n)), 'constant')
        data_e = np.pad(data_e, (0, window_size - len(data_e)), 'constant')
        
    for i in range(0, len(data_z) - window_size + 1, step_size):
        win_z = data_z[i:i+window_size].copy()
        win_n = data_n[i:i+window_size].copy()
        win_e = data_e[i:i+window_size].copy()
        
        max_z = np.max(np.abs(win_z))
        if max_z > 0: win_z = win_z / max_z
        
        max_n = np.max(np.abs(win_n))
        if max_n > 0: win_n = win_n / max_n
            
        max_e = np.max(np.abs(win_e))
        if max_e > 0: win_e = win_e / max_e
            
        emb_3c = np.array([
            utils.latent_codes_1D(win_z.astype(np.float32), model),
            utils.latent_codes_1D(win_n.astype(np.float32), model),
            utils.latent_codes_1D(win_e.astype(np.float32), model)
        ]).reshape(1, -1)
        
        p, _, _ = utils.infer_3C_PDFs(emb_3c, pdf_3C, "Kernel")
        
        if p == 0:
            return 1
            
    return 0

# ==============================================================================
# 🔥 FUNGSI WORKER UNTUK PARALELISASI (SATU THREAD MENGURUS SATU FILE)
# ==============================================================================
def proses_satu_berkas(item, embedding_model, embeddings_3C_PDFs):
    try:
        st = read(item['path'])
        st.detrend("demean")
        st.detrend("linear")
        st.resample(100.0)
        
        comp_z = st.select(component="Z")
        comp_n = st.select(component="N") or st.select(component="1")
        comp_e = st.select(component="E") or st.select(component="2")
        
        if not (comp_z and comp_n and comp_e): 
            return None
        
        idx_start_le, idx_end_le = int(60.0 * 100), int(80.0 * 100)
        idx_start_no, idx_end_no = int(10.0 * 100), int(30.0 * 100)
        
        le_z_block = comp_z[0].data[idx_start_le:idx_end_le]
        le_n_block = comp_n[0].data[idx_start_le:idx_end_le]
        le_e_block = comp_e[0].data[idx_start_le:idx_end_le]
        
        no_z_block = comp_z[0].data[idx_start_no:idx_end_no]
        no_n_block = comp_n[0].data[idx_start_no:idx_end_no]
        no_e_block = comp_e[0].data[idx_start_no:idx_end_no]
        
        keputusan_le = sliding_window_inference_3c(le_z_block, le_n_block, le_e_block, embedding_model, embeddings_3C_PDFs)
        keputusan_no = sliding_window_inference_3c(no_z_block, no_n_block, no_e_block, embedding_model, embeddings_3C_PDFs)
        
        return {
            'y_true': [1, 0],
            'y_pred': [keputusan_le, keputusan_no],
            'rows': [
                {'File': item['eid'], 'Type': 'LE_3C_Scrolling', 'Decision': keputusan_le},
                {'File': item['eid'], 'Type': 'NO_3C_Scrolling', 'Decision': keputusan_no}
            ]
        }
    except Exception:
        return None

# ==============================================================================
# MODUL VISUALISASI (Sama seperti sebelumnya, disembunyikan untuk keringkasan)
# ==============================================================================
# ... (Masukkan fungsi plot_paper_style_cm, plot_bar_metrics_paper_style, visualisasi_hasil_riset di sini)

def plot_bar_metrics_paper_style(cm):
    pass # Pastikan Anda menyalin fungsi visualisasi dari kode sebelumnya ke sini

# ==============================================================================
# 🚀 CORE PROCESS DENGAN MULTITHREADING
# ==============================================================================
def jalankan_stress_test_3C_multi_worker():
    print("\n" + "="*90)
    print("🚀 STARTING: MULTI-WORKER STRESS-TEST 3C (SCROLLING WINDOW)")
    print("="*90)
    
    embedding_model = keras.models.load_model(filepath=MODEL_PATH, compile=False)
    
    print("⏳ Memuat database statistik Typical Embeddings 3C...")
    embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    embedding_N = dataset.load_embedding_data(EMB_DIR, "Embedding data, N.json")
    embedding_E = dataset.load_embedding_data(EMB_DIR, "Embedding data, E.json")
    embeddings_3C_PDFs = utils.embedding_PDFs_3D(embedding_Z, embedding_N, embedding_E)
    
    daftar_file_mseed = []
    for root, _, files in os.walk(OUTPUT_WAVEFORM_DIR):
        for file in files:
            if file.endswith('.mseed'):
                parts = file.replace('.mseed', '').split('_')
                if len(parts) >= 3:
                    daftar_file_mseed.append({'path': os.path.join(root, file), 'eid': parts[2]})

    y_true_list, y_pred_list, report_rows = [], [], []

    # 🔥 IMPLEMENTASI MAX WORKERS (MULTI-THREADING)
    # Gunakan jumlah core CPU maksimal, sisakan 1-2 core agar Mac tidak lag
    MAX_THREADS = min(32, (os.cpu_count() or 1) + 4) 
    print(f"\n⚡ Mengaktifkan Akselerator: Menjalankan {MAX_THREADS} Workers secara paralel...")
    
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        # Kirim semua tugas ke kolam pekerja (worker pool)
        futures = {executor.submit(proses_satu_berkas, item, embedding_model, embeddings_3C_PDFs): item for item in daftar_file_mseed}
        
        # Tangkap hasilnya saat sudah selesai
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing 3C (Fast Mode)"):
            hasil = future.result()
            if hasil is not None:
                y_true_list.extend(hasil['y_true'])
                y_pred_list.extend(hasil['y_pred'])
                report_rows.extend(hasil['rows'])

    if not y_true_list: return
        
    print("\n📊" + "="*34 + " 3C RESULTS (MULTI-WORKER) " + "="*34)
    pd.DataFrame(report_rows).to_csv(PATH_FINAL_REPORT, index=False)
    
    # ... (Panggil fungsi visualisasi Anda di sini)
        
    del embedding_model, y_true_list, y_pred_list
    gc.collect()

if __name__ == "__main__":
    jalankan_stress_test_3C_multi_worker()